In [18]:
import numpy as np
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd
import gzip

from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [20]:
def get_labels(path):
    with gzip.open(path, 'rb') as data:
        labels = data.read()[8:]
        return np.frombuffer(labels, dtype=np.uint8)

def get_images(path):
    with gzip.open(path, 'rb') as data:
        _ = int.from_bytes(data.read(4), 'big')
        num_images = int.from_bytes(data.read(4), 'big')
        rows = int.from_bytes(data.read(4), 'big')
        cols = int.from_bytes(data.read(4), 'big')
        images = data.read()
        return np.frombuffer(images, dtype=np.uint8).reshape((num_images, rows, cols))

In [21]:
mnist_path = '.'

x_trainval = get_images(Path(mnist_path) / 'train-images-idx3-ubyte.gz')
y_trainval = get_labels(Path(mnist_path) / 'train-labels-idx1-ubyte.gz')
x_test = get_images(Path(mnist_path) / 't10k-images-idx3-ubyte.gz')
y_test = get_labels(Path(mnist_path) / 't10k-labels-idx1-ubyte.gz')

x_train = x_trainval[:50000].reshape(50000, -1).astype(np.float32) / 255.0
y_train = y_trainval[:50000]
x_val = x_trainval[50000:].reshape(10000, -1).astype(np.float32) / 255.0
y_val = y_trainval[50000:]
x_test = x_test.reshape(10000, -1).astype(np.float32) / 255.0

print("Train: ", x_train.shape)
print("Test: ", x_test.shape)
print("Val: ", x_val.shape)

Train:  (50000, 784)
Test:  (10000, 784)
Val:  (10000, 784)


In [22]:
class MNISTDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).to(device)
        self.y = torch.tensor(y, dtype=torch.long).to(device)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_data = MNISTDataset(x_train, y_train)
val_data = MNISTDataset(x_val, y_val)
test_data = MNISTDataset(x_test, y_test)